# 📘 W5 Python Lab — 스케줄링 + Vector DB 기초

**n8n W5 강의의 Python 버전.** 멀티 스케줄 + Discord 멀티채널 + Vector DB 첫 문서 저장.

## 학습 목표
- `schedule` 라이브러리로 시장 시간대별 루틴 자동 실행
- Supabase Python 클라이언트로 pgvector 연결
- OpenAI Embedding으로 텍스트 → 벡터 변환
- 첫 문서 저장 + 유사도 검색 테스트

## 🛠 사전 준비

```bash
pip install schedule supabase openai python-dotenv
```

`.env` 추가:
```
SUPABASE_URL=https://xxxxx.supabase.co
SUPABASE_KEY=여러분의_service_role_키
OPENAI_API_KEY=여러분의_OpenAI_키
```

> 💡 Qdrant 선택 시: `pip install qdrant-client` + 본 노트북 Supabase 부분을 Qdrant 예제로 교체.

## 1. 환경 로드 + 클라이언트 초기화

In [ ]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from supabase import create_client
from openai import OpenAI

load_dotenv()

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

# 클라이언트 생성
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

print(f'Supabase: {"✓" if SUPABASE_URL else "✗"}')
print(f'OpenAI: {"✓" if OPENAI_API_KEY else "✗"}')

## 2. Supabase SQL 준비 (SQL Editor에서 1회만 실행)

이 SQL을 Supabase 대시보드의 SQL Editor에서 실행하세요:

```sql
-- 1. pgvector 확장 활성화
CREATE EXTENSION IF NOT EXISTS vector;

-- 2. 문서 테이블 (1536차원 = OpenAI text-embedding-3-small)
CREATE TABLE IF NOT EXISTS documents (
  id uuid PRIMARY KEY DEFAULT gen_random_uuid(),
  content text,
  metadata jsonb,
  embedding vector(1536),
  created_at timestamptz DEFAULT now()
);

-- 3. 유사도 검색 RPC 함수
CREATE OR REPLACE FUNCTION match_documents(
  query_embedding vector(1536),
  match_count int DEFAULT 5
)
RETURNS TABLE (id uuid, content text, metadata jsonb, similarity float)
LANGUAGE plpgsql AS $$
BEGIN
  RETURN QUERY
  SELECT d.id, d.content, d.metadata,
         1 - (d.embedding <=> query_embedding) AS similarity
  FROM documents d
  ORDER BY d.embedding <=> query_embedding
  LIMIT match_count;
END; $$;
```

연결 테스트:

In [ ]:
def test_supabase_connection():
    """Supabase 연결 + 테이블 존재 확인."""
    try:
        result = supabase.table('documents').select('*').limit(1).execute()
        count = len(result.data)
        print(f'✓ documents 테이블 접근 성공 (현재 {count}+개 행)')
        return True
    except Exception as e:
        print(f'✗ 연결 실패: {e}')
        print('→ SQL Editor에서 위 SQL을 실행했는지 확인하세요')
        return False

test_supabase_connection()

## 3. OpenAI Embedding — 텍스트 → 벡터

`text-embedding-3-small`: 100만 토큰 = $0.02. 매우 저렴.

In [ ]:
def get_embedding(text: str) -> list[float]:
    """텍스트를 1536차원 벡터로 변환."""
    # 개행·공백 정리
    text = text.replace('\n', ' ').strip()
    
    response = openai_client.embeddings.create(
        model='text-embedding-3-small',
        input=text
    )
    return response.data[0].embedding

# 테스트
sample = '연준이 금리를 5.5%로 동결했다'
vec = get_embedding(sample)
print(f'벡터 길이: {len(vec)}')
print(f'앞 5개 값: {[round(v, 4) for v in vec[:5]]}')

## 4. 문서 저장 (Insert)

W6 본격 RAG의 기초. 이번엔 간단하게 문장 하나씩 저장.

In [ ]:
def insert_document(content: str, metadata: dict = None) -> str:
    """문서를 Vector DB에 저장."""
    embedding = get_embedding(content)
    
    row = {
        'content': content,
        'metadata': metadata or {},
        'embedding': embedding
    }
    
    result = supabase.table('documents').insert(row).execute()
    inserted_id = result.data[0]['id']
    print(f'✓ 저장 완료: {content[:50]}... → id={inserted_id[:8]}')
    return inserted_id

# 테스트 문장 5개 저장
test_docs = [
    ('FOMC가 3월 회의에서 기준금리를 5.50%로 동결했다',
     {'source': 'FOMC_202503', 'page': 1, 'category': 'fomc'}),
    ('점도표에서 2026년 말 중간값은 3.625%로 제시됐다',
     {'source': 'FOMC_202503', 'page': 4, 'category': 'fomc'}),
    ('애플이 WWDC에서 새로운 AI 칩 M5를 발표했다',
     {'source': 'News_Apple_WWDC', 'page': 1, 'category': 'news'}),
    ('삼성전자 3분기 영업이익이 9.2조원으로 전년 대비 277% 증가했다',
     {'source': 'Earnings_Samsung_3Q', 'page': 2, 'category': 'earnings'}),
    ('한국은행이 기준금리를 3.5%로 7개월 연속 동결했다',
     {'source': 'BOK_Oct_2025', 'page': 1, 'category': 'macro'})
]

for content, meta in test_docs:
    insert_document(content, meta)
    time.sleep(0.3)  # Rate limit 방지

## 5. 유사도 검색 (Query)

n8n의 Vector Store Retrieve = 아래 `search_documents()`.

In [ ]:
def search_documents(query: str, top_k: int = 3) -> list[dict]:
    """질문을 임베딩 후 유사 문서 Top-K 검색."""
    query_vec = get_embedding(query)
    
    # Supabase RPC 호출 (SQL에서 만든 match_documents 함수)
    result = supabase.rpc('match_documents', {
        'query_embedding': query_vec,
        'match_count': top_k
    }).execute()
    
    return result.data

# 테스트 1: FOMC 관련
print('\n=== Q: "연준이 금리를 어떻게 결정했나?" ===')
for doc in search_documents('연준이 금리를 어떻게 결정했나?'):
    print(f'  • [{doc["similarity"]:.3f}] {doc["content"]}')
    print(f'    출처: {doc["metadata"].get("source")}')

# 테스트 2: 한국 경제 관련
print('\n=== Q: "한국은행의 최근 정책은?" ===')
for doc in search_documents('한국은행의 최근 정책은?'):
    print(f'  • [{doc["similarity"]:.3f}] {doc["content"]}')
    print(f'    출처: {doc["metadata"].get("source")}')

# 테스트 3: 애플
print('\n=== Q: "애플 신제품 뭐 나왔어?" ===')
for doc in search_documents('애플 신제품 뭐 나왔어?'):
    print(f'  • [{doc["similarity"]:.3f}] {doc["content"]}')
    print(f'    출처: {doc["metadata"].get("source")}')

## 6. 멀티 스케줄 — 한국장/미국장/크립토 루틴

n8n Schedule Trigger 4개 = Python `schedule` 라이브러리.

In [ ]:
import schedule

# 가짜 에이전트 함수들 (실제론 W4 에이전트 호출)
def kr_morning_routine():
    print(f'[{datetime.now():%H:%M}] 🇰🇷 한국장 개장 브리핑 실행')
    # 실제로는 run_3d_pipeline('삼성전자', 'KRX:005930')

def kr_close_routine():
    print(f'[{datetime.now():%H:%M}] 🇰🇷 한국장 마감 리포트 실행')

def us_open_routine():
    print(f'[{datetime.now():%H:%M}] 🇺🇸 미국장 개장 브리핑 실행')

def crypto_check():
    print(f'[{datetime.now():%H:%M}] 🪙 암호화폐 4시간 체크')

# 스케줄 등록 (실제 Cron 시간 기준)
schedule.every().monday.at('08:30').do(kr_morning_routine)
schedule.every().tuesday.at('08:30').do(kr_morning_routine)
schedule.every().wednesday.at('08:30').do(kr_morning_routine)
schedule.every().thursday.at('08:30').do(kr_morning_routine)
schedule.every().friday.at('08:30').do(kr_morning_routine)

schedule.every().monday.at('15:45').do(kr_close_routine)
schedule.every().monday.at('22:00').do(us_open_routine)

schedule.every(4).hours.do(crypto_check)

print('등록된 작업:')
for job in schedule.get_jobs():
    print(f'  • {job}')

# 실제 운영시: 아래 while 루프 활성화
# while True:
#     schedule.run_pending()
#     time.sleep(60)

## 7. Discord 멀티채널 발송

n8n Switch 노드 = Python if-elif-else. verdict에 따라 다른 Webhook 호출.

In [ ]:
import requests

# 채널별 Webhook (실제로는 .env에서 3개 분리 관리)
CHANNELS = {
    'daily': os.getenv('DISCORD_DAILY_WEBHOOK', ''),
    'urgent': os.getenv('DISCORD_URGENT_WEBHOOK', ''),
    'debug': os.getenv('DISCORD_DEBUG_WEBHOOK', '')
}

def route_notification(verdict: dict):
    """verdict에 따라 적절한 채널로 발송."""
    action = verdict.get('verdict')
    conf = verdict.get('confidence', 0)
    
    if action in ['BUY', 'SELL'] and conf >= 4:
        target = 'urgent'  # 긴급 — 모바일 푸시
        message = f'🚨 **긴급** {action} (신뢰도 {conf}/5)'
    elif action == 'WATCH':
        target = 'daily'  # 일반 채널
        message = f'👀 관찰 대상: {action}'
    else:
        target = 'debug'  # 로그만
        message = f'📋 로그: {action} (conf {conf})'
    
    webhook = CHANNELS.get(target)
    if webhook:
        requests.post(webhook, json={'content': message}, timeout=5)
        print(f'→ {target} 채널로 발송')
    else:
        print(f'→ {target} 채널 미설정, 스킵')

# 테스트
test_verdicts = [
    {'verdict': 'BUY', 'confidence': 5},
    {'verdict': 'WATCH', 'confidence': 3},
    {'verdict': 'HOLD', 'confidence': 2}
]
for v in test_verdicts:
    print(f'\n{v} →')
    route_notification(v)

## 🎯 과제

1. **관심 뉴스 10건 저장** — 본인이 관심 있는 종목/이슈 관련 뉴스 10건을 `insert_document()`로 저장
2. **상위 3개 검색 테스트** — 다양한 질문으로 `search_documents()` 호출 → 의미 검색 품질 체감
3. **metadata 필터 추가** — 아래 SQL로 category 필터 가능한 새 RPC 생성
   ```sql
   CREATE OR REPLACE FUNCTION match_documents_filtered(
     query_embedding vector(1536),
     filter_category text,
     match_count int
   ) RETURNS TABLE (...) 
   ```
4. **스케줄 실제 가동** — 노트북 마지막 셀 주석 풀고 `while True:` 루프 활성화 → 백그라운드 실행 (Ctrl+C로 중단)
5. **비용 측정** — 10건 저장 + 50회 검색 시 예상 비용 계산 (≈ $0.001 미만)

## 🔄 n8n vs Python 비교

| 개념 | n8n | Python |
|---|---|---|
| Schedule Trigger | Cron 설정 | `schedule.every().day.at('...')` |
| Vector Store Insert | Supabase Vector Store 노드 | `supabase.table().insert()` |
| Embedding | Embeddings OpenAI sub-node | `openai_client.embeddings.create()` |
| Vector Search | Retrieve mode | `supabase.rpc('match_documents')` |
| Switch 분기 | Switch 노드 | `if-elif-else` |
| 멀티채널 발송 | Discord 노드 × N | `requests.post()` × N |

**핵심 배움:** Vector DB는 `SELECT ... ORDER BY embedding <=> query`라는 특수한 SQL 쿼리로 작동합니다. pgvector가 PostgreSQL에 **벡터 거리 연산자 `<=>`**를 추가한 확장 패키지입니다. 의미 검색이 마법처럼 보이지만 실체는 **코사인 유사도 계산**일 뿐.